# RSVT — M3Dsynth Training and Results (coarse-to-fine)

This notebook prepares the **M3Dsynth** dataset (manipulated LIDC-IDRI lung CT: nodule removal and insertion by CycleGAN, pix2pix and diffusion generators), checks it, trains the coarse-to-fine RSVT and shows its results.

**Open this notebook in Google Colab from GitHub and choose a GPU runtime** (Runtime → Change runtime type). The project code is cloned from GitHub at `CODE_REF`, and the exact commit is saved with each session. Every output saves automatically to project Drive and is read back against a checksum.

**Data preparation downloads data only when you set `PREPARE_DATA = True`.** With the default patient caps it fetches about 260 LIDC-IDRI series from the public TCIA API (~30 GB) and streams the three author archives (cycle 171 GB, pix2pix 142 GB, diffusion 161 GB) from grip.unina.it. Only verified boxes around each edit are kept, so Drive storage stays near 15–20 GB. Streaming takes hours; it resumes where it stopped.

Licences: M3Dsynth (GRIP-UNINA, non-profit research use with attribution) and LIDC-IDRI (TCIA, CC BY 3.0). Test patients are not downloaded or opened here.


## 1. Choose the run settings

Why: keep the settings you may change in one place. `CODE_REF` chooses the code: a branch, a tag or a commit. To resume a session, paste its Drive folder into `RESUME_SESSION` and keep the same settings and code. `LIDC_ROOT` may point to an existing LIDC-IDRI download in Drive (NBIA folder layout); leave it empty to fetch series from TCIA.


In [ ]:
PROJECT = "/content/drive/MyDrive/Research/LUNG-CT-SCAN"
REPOSITORY = "https://github.com/Richmiz/Risk-Aware-Sparse-Volumetric-Transformer-for-Tampered-Lung-CT-Detection.git"
CODE_REF = "main"         # branch, tag or commit; the exact commit is saved with the session
PREPARE_DATA = False      # True = download/stream and verify M3Dsynth (resumable)
RUN_DIAGNOSTICS = True    # resolution and shortcut checks before training
RUN_TRAINING = True
GENERATORS = ["cycle", "pix2pix", "diffusion"]  # remove one to hold it out
PATIENTS_TRAIN = 200      # of 488 publisher training patients
PATIENTS_VALIDATION = 60  # of 100 publisher validation patients
EPOCHS = 30
LEARNING_RATE = 0.0002
EARLY_STOPPING_PATIENCE = 8
SEED = 13
LIDC_ROOT = ""            # optional existing LIDC-IDRI folder in Drive
RESUME_SESSION = ""       # empty = a new session


## 2. Connect Google Drive

Why: the dataset store and the results live here. Use the Google account containing Research/LUNG-CT-SCAN.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


## 3. Get the project code from GitHub

Why: the implementation lives in the repository, not in long cells. This clones it into the runtime, checks out `CODE_REF` and installs the two missing packages. The repository is public, so no sign-in is needed. Expected output: the exact commit being run.


In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
CODE_ROOT = Path("/content/rsvt-code")
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")

def git(*args):
    done = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr.strip()[-300:]}")
    return done.stdout.strip()

if not (CODE_ROOT / ".git").exists():
    git("clone", "--quiet", REPOSITORY, str(CODE_ROOT))
git("-C", str(CODE_ROOT), "fetch", "--quiet", "--tags", "origin")
commit = None
for candidate in (f"origin/{CODE_REF}", CODE_REF):
    try:
        commit = git("-C", str(CODE_ROOT), "rev-parse", "--verify", "--quiet", candidate + "^{commit}")
        break
    except RuntimeError:
        pass
if commit is None:
    raise RuntimeError(f"CODE_REF {CODE_REF!r} is not a branch, tag or commit of the repository")
if "rsvt" in sys.modules and globals().get("RSVT_COMMIT") not in (None, commit):
    raise RuntimeError("This session already loaded other code. Runtime > Restart session, then Run all.")
git("-C", str(CODE_ROOT), "checkout", "--quiet", "--detach", commit)
RSVT_COMMIT = commit
for module, requirement in (("pydicom", "pydicom==3.0.2"), ("SimpleITK", "SimpleITK==2.5.7")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", requirement], check=True)
if str(CODE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(CODE_ROOT / "src"))
print("Project code at commit", commit)


## 4. Create a saved session

Why: each run gets its own Drive folder with its settings and code commit. The M3Dsynth metadata tables are downloaded from the authors' GitHub at a pinned commit and checked against their recorded checksums. Expected output: the session folder in Drive.


In [ ]:
from rsvt.training.m3d_notebook import create_session
session = create_session(PROJECT, CODE_ROOT, epochs=EPOCHS, learning_rate=LEARNING_RATE,
    patience=EARLY_STOPPING_PATIENCE, seed=SEED, patients_train=PATIENTS_TRAIN,
    patients_validation=PATIENTS_VALIDATION, generators=tuple(GENERATORS),
    lidc_root=LIDC_ROOT or None, resume_session=RESUME_SESSION, code_commit=RSVT_COMMIT)
print("Session saved in Drive:", session.root)


## 5. Check the actual runtime and GPU

Why: a server label is not proof of a GPU. This checks what PyTorch sees. Data preparation runs on CPU; training needs CUDA.


In [ ]:
runtime_information = session.show_runtime()


## 6. Review the data plan before downloading

Why: see patients, series, archive sizes and expected Drive storage before any transfer. Nothing is downloaded here.


In [ ]:
data_plan = session.plan_data()


## 7. Prepare and verify M3Dsynth

Why: build the dataset store. Each manipulated slice is compared with the real LIDC slice while it streams; a sample is kept only if every change lies inside its stored box and the author mask is complete. With `PREPARE_DATA=False` this cell only verifies an existing store. Expected output: progress, then a checksum-verified manifest.


In [ ]:
manifest = session.prepare_data(download=PREPARE_DATA)


## 8. Inspect the dataset with real CT slices

Why: confirm what the model will see. Expected output: counts per partition/generator/operation and training examples showing the authentic slice, the manipulated slice, their difference and the author mask. No test data is opened.


In [ ]:
if session.has_store():
    dataset_overview = session.show_dataset()
    preview_examples = session.show_previews()


## 9. Pre-training checks: resolution and shortcuts

Why: test the review findings on real data before spending GPU time. (a) A small CNN separates manipulated crops from authentic sites that look alike (removals vs healthy-tissue TB sites, insertions vs real-nodule TM sites) at native resolution and degraded to the 96³ voxel size; results are shown per operation. This tests the evidence at a known site, not whether the router can find it. (b) Archive identity outside the edit, mask coverage, and whether whole-scan statistics alone separate classes. Uses train/validation only.


In [ ]:
if RUN_DIAGNOSTICS and session.has_store():
    diagnostics = session.run_diagnostics()


## 10. Understand the training configuration

Why: see batches, crops, loss and risk weights, early stopping and checkpoint selection before training.


In [ ]:
configuration_table = session.show_configuration()


## 11. Build and inspect the coarse-to-fine RSVT

Why: see parameters and compute. The 96³ view gives anatomy context and a router chooses K native-resolution crops; evidence is read only there. The table compares this with running the same encoder over the whole native volume.


In [ ]:
if session.has_store():
    parameter_count = session.show_model()


## 12. Train and watch the epoch results

Why: learn from training volumes and monitor validation volumes. Expected output: batch progress, an updating epoch table and learning curves. The first epochs train without tail risk; the checkpoint is chosen among risk-stage epochs by the validation loss averaged over authentic, removal and insertion volumes. Checkpoints and logs save to Drive after every epoch; re-running resumes.


In [ ]:
if RUN_TRAINING and session.has_store():
    training_summary = session.train()
else:
    print("Training not run in this execution. No training result is claimed.")


## 13. Review the learning curves

Why: compare training objective with validation loss and AUROCs. Falling training loss alone does not prove generalization.


In [ ]:
epoch_history = session.show_history()


## 14. Inspect the selected model’s results

Why: judge detection where it matters: removal (FB) and insertion (FM) against authentic scans, per generator, router recall (did the chosen crops contain the edit?) and mask overlap. Operating points are exploratory.


In [ ]:
selected_model_results = session.show_results()


## 15. Understand the held-out test set

Why: keep an independent final check. Test patients are not acquired or opened by this notebook.


In [ ]:
session.show_test_status()


## 16. Verify that the results are saved

Why: confirm Drive storage by reading files back against checksums. Expected output: **Saved and verified** and the session path.


In [ ]:
persistence_receipt = session.verify_saved()
print("To resume this session:", session.root)
